# Regression Models — Advanced
**Prerequisites:** Complete **Regression Models — Basic** first.

This notebook covers non-linear and ensemble regression methods, then shows how to
compare all models and tune hyperparameters with Grid Search and Random Search.

## Setup

Re-running the necessary setup from the Basic notebook in condensed form.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

In [ ]:
column_names = ['CRIM','ZN','INDUS','CHAS','NOX','RM','AGE','DIS','RAD','TAX','PTRATIO','B','LSTAT','MEDV']
df = pd.read_csv('../data/housing.csv', header=None, delimiter=r"\s+", names=column_names)

X = df.drop('MEDV', axis=1)
y = df['MEDV']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test),      columns=X.columns)

def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f"{'─'*40}\n  {name}\n  MAE: {mae:.3f}  RMSE: {rmse:.3f}  R²: {r2:.4f}")

print("Setup complete.")

## ElasticNet

ElasticNet combines Ridge (L2) and Lasso (L1) regularization into a single model:

$$\text{Loss} = \sum(y_i - \hat{y}_i)^2 + \alpha \left[ \frac{1-\rho}{2} \sum \beta_j^2 + \rho \sum |\beta_j| \right]$$

The `l1_ratio` (ρ) controls the mix:
- `l1_ratio = 0` → Pure Ridge
- `l1_ratio = 1` → Pure Lasso
- `0 < l1_ratio < 1` → Mix of both

**Best for:** Datasets with many correlated features where you want some feature selection but not the instability of pure Lasso.

In [ ]:
elastic = ElasticNet(alpha=1.0, l1_ratio=0.5, random_state=42)
elastic.fit(X_train_scaled, y_train)
y_pred_elastic = elastic.predict(X_test_scaled)
evaluate("ElasticNet (alpha=1.0, l1_ratio=0.5)", y_test, y_pred_elastic)

# Count zeroed features
n_zero = (elastic.coef_ == 0).sum()
print(f"  Features zeroed out: {n_zero}/{len(elastic.coef_)}")

In [ ]:
# How does l1_ratio affect coefficient sparsity?
ratios = [0.1, 0.3, 0.5, 0.7, 0.9]
n_zeros = [sum(ElasticNet(alpha=1.0, l1_ratio=r).fit(X_train_scaled, y_train).coef_ == 0)
           for r in ratios]

plt.figure(figsize=(7, 4))
plt.plot(ratios, n_zeros, 'o-', color='purple')
plt.xlabel('l1_ratio (0=Ridge, 1=Lasso)')
plt.ylabel('Number of Zero Coefficients')
plt.title('ElasticNet — Sparsity vs. l1_ratio')
plt.grid(True)
plt.show()

## Support Vector Regression (SVR)

SVR adapts the Support Vector Machine idea to regression. Instead of finding a decision boundary,
it finds a "tube" of width ε around the predictions — errors within the tube are ignored.

$$\min \frac{1}{2}\|\mathbf{w}\|^2 + C \sum \max(0, |y_i - \hat{y}_i| - \varepsilon)$$

Key parameters:
- **C:** Regularization — higher C = less regularization = tighter fit to training data
- **epsilon (ε):** Width of the error tube — errors within ε don't contribute to the loss
- **kernel:** The function used to map data into higher dimensions (`rbf`, `linear`, `poly`)

**Critical:** SVR is sensitive to feature scale, so **always** use it inside a pipeline with a scaler.

In [ ]:
# SVR must always be paired with feature scaling
svr = make_pipeline(StandardScaler(), SVR(C=1.0, epsilon=0.2))
svr.fit(X_train, y_train)
y_pred_svr = svr.predict(X_test)
evaluate("SVR (C=1.0, ε=0.2, rbf kernel)", y_test, y_pred_svr)

In [ ]:
# Comparing kernel choices
kernels = ['linear', 'rbf', 'poly']
for k in kernels:
    m = make_pipeline(StandardScaler(), SVR(kernel=k, C=1.0)).fit(X_train, y_train)
    r2 = r2_score(y_test, m.predict(X_test))
    print(f"  SVR ({k} kernel): R² = {r2:.4f}")

## Decision Tree Regressor

A Decision Tree splits the data into rectangular regions using binary rules (e.g., `RM > 6.5`),
then predicts the mean value of samples in each region (leaf node).

**Strengths:**
- Non-linear — can capture complex patterns
- No scaling required
- Highly interpretable (can be visualised)

**Weaknesses:**
- Prone to **overfitting** — will perfectly memorise training data without constraints
- High variance — small changes in data can produce very different trees

The key hyperparameters to control overfitting: `max_depth`, `min_samples_split`, `min_samples_leaf`.

In [ ]:
from sklearn.tree import DecisionTreeRegressor, plot_tree

# Unlimited depth — likely overfits
dt_full = DecisionTreeRegressor(random_state=42)
dt_full.fit(X_train, y_train)
train_r2 = r2_score(y_train, dt_full.predict(X_train))
test_r2  = r2_score(y_test,  dt_full.predict(X_test))
print(f"Full tree — Train R²: {train_r2:.4f}  Test R²: {test_r2:.4f}  (gap = sign of overfitting)")

# Depth-limited — better generalisation
dt = DecisionTreeRegressor(max_depth=4, random_state=42)
dt.fit(X_train, y_train)
evaluate("Decision Tree (max_depth=4)", y_test, dt.predict(X_test))

In [ ]:
# Visualise the tree (limited depth for readability)
plt.figure(figsize=(20, 6))
plot_tree(dt, feature_names=X.columns, filled=True, rounded=True, fontsize=9)
plt.title('Decision Tree Regressor (max_depth=4)')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance from the tree
importances = pd.Series(dt.feature_importances_, index=X.columns).sort_values(ascending=True)
plt.figure(figsize=(8, 5))
importances.plot(kind='barh', color='steelblue')
plt.title('Decision Tree — Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## Random Forest Regressor

Random Forest is an **ensemble** of Decision Trees using **Bagging** (Bootstrap Aggregating):

1. Create `n_estimators` bootstrap samples from training data (random subsets with replacement)
2. Fit one Decision Tree on each sample, using a random subset of features at each split
3. Prediction = average of all trees' predictions

This reduces **variance** (the main weakness of individual trees) while keeping low bias.
The randomness means each tree is different — their errors partially cancel out.

**Out-of-Bag (OOB) Error:** Because each tree is trained on only ~63% of samples,
the remaining ~37% (out-of-bag) can be used as a free validation set.

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42, oob_score=True, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

evaluate("Random Forest (100 trees)", y_test, y_pred_rf)
print(f"  OOB R² Score: {rf.oob_score_:.4f}  (free validation without a separate val set)")

In [ ]:
# Feature importances from Random Forest (averaged across all trees — more reliable than single tree)
importances_rf = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)
plt.figure(figsize=(8, 5))
importances_rf.plot(kind='barh', color='steelblue')
plt.title('Random Forest — Feature Importances (avg across 100 trees)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## Gradient Boosting Regressor

Gradient Boosting is another ensemble method, but unlike Random Forest (parallel trees),
it builds trees **sequentially**. Each tree corrects the residual errors of the previous one:

1. Start with a simple prediction (e.g., mean of y)
2. Compute residuals (errors) between predictions and actual values
3. Fit a shallow tree to predict those residuals
4. Add the tree's predictions to the current model (scaled by `learning_rate`)
5. Repeat for `n_estimators` rounds

**Strengths:** Often the best-performing model on tabular data.  
**Weaknesses:** More hyperparameters to tune; slower to train than Random Forest.

The `learning_rate` controls how much each tree contributes — smaller rate needs more trees but often generalises better.

In [ ]:
gb = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)

evaluate("Gradient Boosting (100 trees, lr=0.1)", y_test, y_pred_gb)

In [ ]:
# Learning curve: how many trees do we actually need?
train_errors, test_errors = [], []
for pred in gb.staged_predict(X_train):
    train_errors.append(np.sqrt(mean_squared_error(y_train, pred)))
for pred in gb.staged_predict(X_test):
    test_errors.append(np.sqrt(mean_squared_error(y_test, pred)))

plt.figure(figsize=(8, 4))
plt.plot(train_errors, label='Train RMSE', color='steelblue')
plt.plot(test_errors, label='Test RMSE', color='tomato')
plt.xlabel('Number of Trees')
plt.ylabel('RMSE')
plt.title('Gradient Boosting — Learning Curve')
plt.legend()
plt.grid(True)
plt.show()

## Full Model Comparison

Let's compare all models we've covered across both notebooks.

In [ ]:
all_models = {
    'Linear Regression':    (LinearRegression(),                         X_train,        X_test),
    'Ridge (α=1)':          (Ridge(alpha=1.0),                           X_train_scaled, X_test_scaled),
    'Lasso (α=1)':          (Lasso(alpha=1.0),                           X_train_scaled, X_test_scaled),
    'ElasticNet':           (ElasticNet(random_state=42),                X_train_scaled, X_test_scaled),
    'SVR':                  (make_pipeline(StandardScaler(), SVR()),      X_train,        X_test),
    'Decision Tree':        (DecisionTreeRegressor(max_depth=4, random_state=42), X_train, X_test),
    'Random Forest':        (RandomForestRegressor(random_state=42, n_jobs=-1), X_train, X_test),
    'Gradient Boosting':    (GradientBoostingRegressor(random_state=42), X_train,        X_test),
}

rows = []
for name, (model, Xtr, Xte) in all_models.items():
    model.fit(Xtr, y_train)
    pred = model.predict(Xte)
    rows.append({'Model': name,
                 'MAE':  round(mean_absolute_error(y_test, pred), 3),
                 'RMSE': round(np.sqrt(mean_squared_error(y_test, pred)), 3),
                 'R²':   round(r2_score(y_test, pred), 4)})

comp = pd.DataFrame(rows).set_index('Model').sort_values('R²', ascending=False)
print(comp)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

comp[['MAE', 'RMSE']].plot(kind='bar', ax=axes[0], rot=30)
axes[0].set_title('Error Metrics — lower is better')
axes[0].set_ylabel('Score')
axes[0].grid(axis='y')

comp[['R²']].plot(kind='bar', ax=axes[1], color='steelblue', rot=30, legend=False)
axes[1].set_title('R² Score — higher is better')
axes[1].set_ylabel('R²')
axes[1].set_ylim(0, 1)
axes[1].grid(axis='y')

plt.suptitle('All Regression Models — Comparison', fontsize=13)
plt.tight_layout()
plt.show()

## Hyperparameter Tuning

A **hyperparameter** is a setting you choose *before* training (e.g., `max_depth`, `learning_rate`).
Unlike model parameters (learned from data), hyperparameters must be set manually or searched.

We'll tune Gradient Boosting using two strategies:
1. **Grid Search** — exhaustively tries every combination in a grid
2. **Random Search** — randomly samples combinations (faster, often finds similar results)

Both use **cross-validation** (5-fold) to evaluate each combination fairly.

### Grid Search

Exhaustive — evaluates every possible combination. Good when the grid is small.

The `neg_mean_squared_error` scoring is used because sklearn *maximises* the score, so we negate MSE (lower MSE = higher score).

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5],
    'min_samples_split': [2, 4],
}

grid_search = GridSearchCV(
    estimator=GradientBoostingRegressor(random_state=42),
    param_grid=param_grid,
    scoring='neg_mean_squared_error',
    cv=5,
    n_jobs=-1,
    verbose=1,
)

t0 = time.time()
grid_search.fit(X_train, y_train)
print(f"\nGrid Search completed in {time.time()-t0:.1f}s")
print(f"Best params: {grid_search.best_params_}")
print(f"Best CV MSE: {-grid_search.best_score_:.3f}")

y_pred_gs = grid_search.predict(X_test)
evaluate("Gradient Boosting (Grid Search tuned)", y_test, y_pred_gs)

joblib.dump(grid_search.best_estimator_, 'best_model_GS.joblib')

### Random Search

Samples `n_iter` random combinations. Much faster for large grids, especially with continuous distributions.

In [ ]:
from scipy.stats import randint, uniform

param_dist = {
    'n_estimators': randint(50, 300),
    'learning_rate': uniform(0.01, 0.3),
    'max_depth': randint(2, 7),
    'min_samples_split': randint(2, 10),
}

random_search = RandomizedSearchCV(
    estimator=GradientBoostingRegressor(random_state=42),
    param_distributions=param_dist,
    n_iter=30,
    scoring='neg_mean_squared_error',
    cv=5,
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

t0 = time.time()
random_search.fit(X_train, y_train)
print(f"\nRandom Search completed in {time.time()-t0:.1f}s")
print(f"Best params: {random_search.best_params_}")

y_pred_rs = random_search.predict(X_test)
evaluate("Gradient Boosting (Random Search tuned)", y_test, y_pred_rs)

joblib.dump(random_search.best_estimator_, 'best_model_RS.joblib')

In [ ]:
# Compare tuned vs untuned
print("\nImpact of hyperparameter tuning on R²:")
print(f"  Untuned Gradient Boosting: {r2_score(y_test, y_pred_gb):.4f}")
print(f"  Grid Search tuned:         {r2_score(y_test, y_pred_gs):.4f}")
print(f"  Random Search tuned:       {r2_score(y_test, y_pred_rs):.4f}")

## Summary

| Model | Type | Needs Scaling | Tuning Priority |
|-------|------|---------------|-----------------|
| ElasticNet | Linear | Yes | alpha, l1_ratio |
| SVR | Kernel | Yes | C, epsilon, kernel |
| Decision Tree | Non-linear | No | max_depth |
| Random Forest | Ensemble (Bagging) | No | n_estimators, max_features |
| Gradient Boosting | Ensemble (Boosting) | No | n_estimators, learning_rate, max_depth |

**General advice:** Start with Linear Regression as a baseline → try Random Forest → try Gradient Boosting with tuning.